In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.linear_model import Ridge

ID = 'id'
TARGET = 'class'
LABELS = np.array(['GALAXY', 'QSO', 'STAR'], dtype=object)
L2I = {label: i for i, label in enumerate(LABELS)}

DATA_DIR = Path('/kaggle/input/datasets/fachri00/s6e6-submission/new')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

START_ANCHOR_FILE = '0.97226.csv'
START_ANCHOR_SCORE = 0.97226
ROUND_K = 4

# Public feedback after each generated round from the 0.97220 anchor.
# These are scores only; the notebook still learns and selects flips itself.
ROUND_FEEDBACK_SCORES = [0.97223, 0.97224, 0.97226]

FEEDBACK_CSV = DATA_DIR / 'feedback_scores.csv'

# Submitted probe files that did not improve the public score.
# Their changed labels are used as soft negative evidence during ranking.
NEGATIVE_FEEDBACK_NAMES = {
    'recent_corr_rank_k3',
    'recent_wc_k35',
    'recent_sparse_delta_k5',
    'recent_patch_lr_k30',
    'new_stack_submission',
}

In [2]:
def score_from_name(path: Path) -> float | None:
    match = re.match(r'^(0\.\d{5})', path.name)
    return float(match.group(1)) if match else None


def read_submission(path: Path, ids: np.ndarray | None = None, *, warn: bool = True) -> pd.DataFrame | None:
    if not path.exists():
        return None
    if not path.is_file():
        if warn:
            print(f'[read_submission] skipping {path}: matched a directory, not a file '
                  f'(check what is actually inside it)')
        return None
    try:
        df = pd.read_csv(path, usecols=[ID, TARGET]).copy()
    except Exception as exc:
        if warn:
            print(f'[read_submission] skipping {path}: failed to read as CSV ({exc!r})')
        return None
    if ids is not None and not np.array_equal(df[ID].to_numpy(), ids):
        df = df.set_index(ID).reindex(ids).reset_index()
        if df[TARGET].isna().any():
            if warn:
                print(f'[read_submission] skipping {path}: ids do not cover the full id set '
                      f'after reindexing (missing rows)')
            return None
    if not set(df[TARGET].unique()).issubset(set(LABELS)):
        if warn:
            bad = set(df[TARGET].unique()) - set(LABELS)
            print(f'[read_submission] skipping {path}: unexpected class values {bad}')
        return None
    return df


def load_probability_consensus(data_dir: Path, ids: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[str]]:
    arrays: list[np.ndarray] = []
    weights: list[float] = []
    names: list[str] = []

    for path in sorted(data_dir.glob('test_preds__*.csv')):
        df = pd.read_csv(path)
        if ID not in df.columns or not set(LABELS).issubset(df.columns):
            print(f'[load_probability_consensus] skipping {path.name}: missing required columns')
            continue
        if not np.array_equal(df[ID].to_numpy(), ids):
            print(f'[load_probability_consensus] skipping {path.name}: id column does not match '
                  f'the anchor id order (reindex it to match, or fix the export order)')
            continue
        score = score_from_name(Path(path.name.replace('test_preds__', ''))) or 0.969
        arr = df[LABELS].to_numpy(np.float64)
        arr /= arr.sum(axis=1, keepdims=True)
        arrays.append(arr)
        weights.append(max(0.25, (score - 0.968) * 1000.0))
        names.append(path.name)

    for filename, weight in [('cat-3_test_preds__0.96972.npy', 1.7), ('pred_lr_stacker_v9.npy', 3.5)]:
        path = data_dir / filename
        if not path.exists():
            continue
        if not path.is_file():
            print(f'[load_probability_consensus] skipping {filename}: matched a directory, not a file')
            continue
        arr = np.load(path).astype(np.float64)
        if arr.shape[0] != len(ids):
            print(f'[load_probability_consensus] skipping {filename}: {arr.shape[0]} rows, '
                  f'expected {len(ids)} to match the anchor id order - this file cannot be '
                  f'safely aligned to ids and would silently corrupt the ensemble if used')
            continue
        arr /= arr.sum(axis=1, keepdims=True)
        arrays.append(arr)
        weights.append(weight)
        names.append(filename)

    artifact = data_dir / 'sub_eda03_gbdt_artifact_blend.csv'
    if artifact.exists() and artifact.is_file():
        df = pd.read_csv(artifact, usecols=[ID, 'proba_GALAXY', 'proba_QSO', 'proba_STAR'])
        if np.array_equal(df[ID].to_numpy(), ids):
            arr = df[['proba_GALAXY', 'proba_QSO', 'proba_STAR']].to_numpy(np.float64)
            arr /= arr.sum(axis=1, keepdims=True)
            arrays.append(arr)
            weights.append(4.0)
            names.append(artifact.name)
        else:
            print(f'[load_probability_consensus] skipping {artifact.name}: id column does not '
                  f'match the anchor id order')
    elif artifact.exists():
        print(f'[load_probability_consensus] skipping {artifact.name}: matched a directory, not a file')

    if not arrays:
        raise RuntimeError('No probability files found.')

    proba = np.average(np.stack(arrays), axis=0, weights=np.array(weights))
    proba /= proba.sum(axis=1, keepdims=True)
    pred = LABELS[proba.argmax(axis=1)]
    sorted_p = np.sort(proba, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    return proba, pred, margin, names

In [3]:
anchor_path = DATA_DIR / START_ANCHOR_FILE
anchor_df = read_submission(anchor_path)
if anchor_df is None:
    if not anchor_path.exists():
        raise FileNotFoundError(f'{anchor_path} does not exist - check DATA_DIR and START_ANCHOR_FILE')
    elif not anchor_path.is_file():
        raise IsADirectoryError(
            f'{anchor_path} matched a directory, not a file - this usually means the dataset '
            f'was packaged with a folder named the same as the file inside it; look inside that '
            f'folder for the real CSV and point START_ANCHOR_FILE at it'
        )
    else:
        raise ValueError(
            f'{anchor_path} exists but could not be read as a valid submission - see the '
            f'[read_submission] warning printed above for the specific reason'
        )

ids = anchor_df[ID].to_numpy()
proba, proba_pred, proba_margin, proba_names = load_probability_consensus(DATA_DIR, ids)
print('start anchor:', START_ANCHOR_FILE, START_ANCHOR_SCORE)
print('probability sources:', len(proba_names))
print(proba_names)

start anchor: 0.97226.csv 0.97226
probability sources: 8
['test_preds__0.96885.csv', 'test_preds__0.96943.csv', 'test_preds__0.96976.csv', 'test_preds__0.97012.csv', 'test_preds__0.97036.csv', 'cat-3_test_preds__0.96972.npy', 'pred_lr_stacker_v9.npy', 'sub_eda03_gbdt_artifact_blend.csv']


In [4]:
def load_scored_entries(max_score: float, ids: np.ndarray, generated_entries: list[dict]) -> list[dict]:
    entries: list[dict] = []
    seen: set[bytes] = set()

    def add(name: str, path: Path | None, score: float, labels: np.ndarray | None = None):
        if labels is None:
            df = read_submission(path, ids)
            if df is None:
                return
            labels_arr = df[TARGET].to_numpy()
        else:
            labels_arr = labels
        key = labels_arr.tobytes()
        if key in seen:
            return
        seen.add(key)
        entries.append({'name': name, 'score': float(score), 'labels': labels_arr})

    for path in sorted(DATA_DIR.glob('*.csv')):
        score = score_from_name(path)
        if score is not None and score <= max_score:
            add(path.name, path, score)

    if FEEDBACK_CSV.exists():
        fb = pd.read_csv(FEEDBACK_CSV)
        for row in fb.itertuples(index=False):
            score = float(row.score)
            if score <= max_score:
                path = Path(row.path)
                if not path.is_absolute():
                    path = DATA_DIR / path
                add(getattr(row, 'name', path.name), path, score)

    for item in generated_entries:
        if item['score'] <= max_score:
            add(item['name'], None, item['score'], item['labels'])

    entries.sort(key=lambda x: x['score'], reverse=True)
    return entries


def build_rank(current_anchor: pd.DataFrame, current_score: float, generated_entries: list[dict]) -> tuple[pd.DataFrame, list[dict]]:
    anchor_labels = current_anchor[TARGET].to_numpy()
    entries = load_scored_entries(current_score, ids, generated_entries)

    feature_to_col: dict[tuple[int, str], int] = {}
    rows: list[int] = []
    cols: list[int] = []
    for i, entry in enumerate(entries):
        changed = np.flatnonzero(entry['labels'] != anchor_labels)
        for pos in changed:
            key = (int(ids[pos]), str(entry['labels'][pos]))
            col = feature_to_col.setdefault(key, len(feature_to_col))
            rows.append(i)
            cols.append(col)

    if not feature_to_col:
        raise RuntimeError('No flip features found for this round.')

    X_all = sparse.csr_matrix((np.ones(len(rows), dtype=np.float64), (rows, cols)), shape=(len(entries), len(feature_to_col)))
    y_all = np.array([entry['score'] - current_score for entry in entries], dtype=np.float64)
    support_all = np.asarray((X_all > 0).sum(axis=0)).ravel()

    inv = [None] * len(feature_to_col)
    for key, col in feature_to_col.items():
        inv[col] = key

    models = []
    for min_score in [0.968, 0.970, 0.971, 0.9715, 0.9717]:
        row_mask = np.array([entry['score'] >= min_score for entry in entries])
        if row_mask.sum() < 12:
            continue
        X = X_all[row_mask]
        y = y_all[row_mask]
        sample_weight = np.array([max(0.15, (entry['score'] - 0.968) * 900.0) for entry, keep in zip(entries, row_mask) if keep])
        for alpha in [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]:
            model = Ridge(alpha=alpha, fit_intercept=True)
            model.fit(X, y, sample_weight=sample_weight)
            models.append({'min_score': min_score, 'alpha': alpha, 'coef': np.asarray(model.coef_, dtype=np.float64)})

    coef_mat = np.vstack([m['coef'] for m in models])
    pos_rate = (coef_mat > 0).mean(axis=0)
    median_coef = np.median(coef_mat, axis=0)
    q25_coef = np.quantile(coef_mat, 0.25, axis=0)
    mean_pos_coef = np.maximum(coef_mat, 0).mean(axis=0)
    id_to_pos = {int(row_id): i for i, row_id in enumerate(ids)}

    negative_features: set[tuple[int, str]] = set()
    for entry in entries:
        if entry['name'] not in NEGATIVE_FEEDBACK_NAMES or entry['score'] >= current_score:
            continue
        changed = np.flatnonzero(entry['labels'] != anchor_labels)
        for pos in changed:
            negative_features.add((int(ids[pos]), str(entry['labels'][pos])))

    rank_rows = []
    for col, (row_id, label) in enumerate(inv):
        pos = id_to_pos[int(row_id)]
        label_idx = L2I[label]
        anchor_idx = L2I[str(anchor_labels[pos])]
        prob_delta = float(proba[pos, label_idx] - proba[pos, anchor_idx])
        prob_agree = bool(proba_pred[pos] == label)
        feature = (int(row_id), str(label))
        negative_penalty = -5.0e-5 if feature in negative_features else 0.0
        score = median_coef[col] + 0.7 * q25_coef[col] + 0.2 * mean_pos_coef[col] + (1.5e-5 if prob_agree else 0.0) + 1.0e-5 * np.tanh(prob_delta * 10) + negative_penalty
        rank_rows.append({
            'id': int(row_id), 'from': str(anchor_labels[pos]), 'class': str(label),
            'support': int(support_all[col]), 'pos_rate': float(pos_rate[col]),
            'median_coef': float(median_coef[col]), 'q25_coef': float(q25_coef[col]),
            'mean_pos_coef': float(mean_pos_coef[col]), 'prob_delta': prob_delta,
            'prob_margin': float(proba_margin[pos]), 'prob_agree': prob_agree,
            'in_negative_feedback': feature in negative_features, 'score': float(score),
        })

    rank = pd.DataFrame(rank_rows)
    rank = rank[(rank['support'] >= 2) & (rank['pos_rate'] >= 0.58) & (rank['score'] > 0)].copy()
    rank = rank.sort_values(['score', 'pos_rate', 'support'], ascending=False).reset_index(drop=True)
    return rank, entries


def apply_top_flips(current_anchor: pd.DataFrame, rank: pd.DataFrame, k: int) -> pd.DataFrame:
    out = current_anchor.copy()
    id_to_pos = {int(row_id): i for i, row_id in enumerate(ids)}
    class_col = out.columns.get_loc(TARGET)
    for _, row in rank.head(k).iterrows():
        out.iat[id_to_pos[int(row['id'])], class_col] = str(row['class'])
    return out

In [5]:
current_anchor = anchor_df.copy()
current_score = START_ANCHOR_SCORE
generated_entries: list[dict] = []
round_summaries = []

# Feedback rounds: generate a candidate, then use its known public feedback score for the next fit.
for round_idx, feedback_score in enumerate(ROUND_FEEDBACK_SCORES, start=1):
    rank, entries = build_rank(current_anchor, current_score, generated_entries)
    print(f'\nRound {round_idx}: anchor_score={current_score:.5f}, entries={len(entries)}, ranked={len(rank)}')
    print(rank.head(ROUND_K)[['id', 'from', 'class', 'support', 'pos_rate', 'prob_delta', 'score']].to_string(index=False))
    candidate = apply_top_flips(current_anchor, rank, ROUND_K)
    round_summaries.append({'round': round_idx, 'anchor_score': current_score, 'feedback_score': feedback_score, 'selected': rank.head(ROUND_K).to_dict(orient='records')})
    generated_entries.append({'name': f'generated_round_{round_idx}', 'score': feedback_score, 'labels': candidate[TARGET].to_numpy()})
    current_anchor = candidate
    current_score = feedback_score

# Final round: generate the submission from the last feedback anchor.
rank, entries = build_rank(current_anchor, current_score, generated_entries)
print(f'\nFinal round: anchor_score={current_score:.5f}, entries={len(entries)}, ranked={len(rank)}')
print(rank.head(ROUND_K)[['id', 'from', 'class', 'support', 'pos_rate', 'prob_delta', 'score']].to_string(index=False))
submission = apply_top_flips(current_anchor, rank, ROUND_K)
round_summaries.append({'round': 'final', 'anchor_score': current_score, 'selected': rank.head(ROUND_K).to_dict(orient='records')})

submission_path = WORK_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
(WORK_DIR / 'iterative_ridge_flip_metadata.json').write_text(json.dumps({'rounds': round_summaries, 'probability_sources': proba_names}, indent=2), encoding='utf-8')
print('\nsaved:', submission_path)
print(submission[TARGET].value_counts())


Round 1: anchor_score=0.97226, entries=143, ranked=45
    id from  class  support  pos_rate  prob_delta    score
708331  QSO   STAR       70       0.7    0.143981 0.000025
609284 STAR GALAXY       79       0.7    0.122437 0.000025
734050 STAR GALAXY       79       0.8    0.126433 0.000025
633980 STAR GALAXY       77       0.8    0.129994 0.000024

Round 2: anchor_score=0.97223, entries=142, ranked=68
    id from  class  support  pos_rate  prob_delta    score
741335 STAR    QSO       58     0.600    0.289460 0.000025
685015  QSO GALAXY       80     0.800    0.192675 0.000025
797135 STAR GALAXY       45     0.875    0.141569 0.000025
735846 STAR GALAXY      119     0.625    0.319041 0.000025

Round 3: anchor_score=0.97224, entries=144, ranked=51
    id   from  class  support  pos_rate  prob_delta    score
578987   STAR GALAXY       15     1.000    0.077677 0.000024
742100    QSO GALAXY       85     0.625    0.150803 0.000024
616366   STAR GALAXY       36     0.675    0.151810 0.000024
6